In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageFile
import os
import glob
from tqdm import tqdm
import json
import pandas as pd
from datetime import datetime
import warnings
import traceback
import logging
from pathlib import Path
import csv

# Try to import SSIM
try:
    from skimage.metrics import structural_similarity as ssim
except ImportError:
    import subprocess
    subprocess.check_call(["pip", "install", "scikit-image"])
    from skimage.metrics import structural_similarity as ssim

# Enable loading of truncated images
ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings('ignore')
plt.switch_backend('Agg')

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# CONFIGURATION - UPDATE THESE PATHS
MODEL_PATH = "D:/Jafar/New Low-dose/CH/2(50%)/generated/experiment_20251117_134739/checkpoints/best_model.pth"
TEST_INPUT_FOLDER = "D:/Jafar/New Low-dose/CH/2(50%)/test"
TEST_TARGET_FOLDER = "D:/Jafar/New Low-dose/CH/Normal/test"
OUTPUT_FOLDER = "D:/Jafar/New Low-dose/CH/2(50%)/Inference_Results"
IMAGE_SIZE = 256
BATCH_SIZE = 1  # Use batch size 1 for inference
DEVICE = 'auto'

# ============================================================================
# REUSE MODEL CLASSES FROM TRAINING SCRIPT
# ============================================================================

class SSIMLoss(nn.Module):
    """SSIM Loss for medical images"""
    
    def __init__(self, window_size=11, data_range=2.0):
        super(SSIMLoss, self).__init__()
        self.window_size = window_size
        self.data_range = data_range
        self.channel = 3
        self.window = self._create_window(window_size, self.channel)
        
    def _gaussian(self, window_size, sigma):
        gauss = torch.Tensor([
            torch.exp(torch.tensor(-(x - window_size // 2) ** 2 / float(2 * sigma ** 2)))
            for x in range(window_size)
        ])
        return gauss / gauss.sum()
    
    def _create_window(self, window_size, channel):
        _1D_window = self._gaussian(window_size, 1.5).unsqueeze(1)
        _2D_window = _1D_window.mm(_1D_window.t()).float().unsqueeze(0).unsqueeze(0)
        window = _2D_window.expand(channel, 1, window_size, window_size).contiguous()
        return window
    
    def _ssim(self, img1, img2, window, window_size, channel):
        mu1 = F.conv2d(img1, window, padding=window_size // 2, groups=channel)
        mu2 = F.conv2d(img2, window, padding=window_size // 2, groups=channel)
        
        mu1_sq = mu1.pow(2)
        mu2_sq = mu2.pow(2)
        mu1_mu2 = mu1 * mu2
        
        sigma1_sq = F.conv2d(img1 * img1, window, padding=window_size // 2, groups=channel) - mu1_sq
        sigma2_sq = F.conv2d(img2 * img2, window, padding=window_size // 2, groups=channel) - mu2_sq
        sigma12 = F.conv2d(img1 * img2, window, padding=window_size // 2, groups=channel) - mu1_mu2
        
        C1 = (0.01 * self.data_range) ** 2
        C2 = (0.03 * self.data_range) ** 2
        
        ssim_map = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2))
        return ssim_map.mean()
    
    def forward(self, img1, img2):
        (_, channel, _, _) = img1.size()
        
        if channel == self.channel and self.window.data.type() == img1.data.type():
            window = self.window
        else:
            window = self._create_window(self.window_size, channel)
            if img1.is_cuda:
                window = window.cuda(img1.get_device())
            window = window.type_as(img1)
            self.window = window
            self.channel = channel
        
        return 1 - self._ssim(img1, img2, window, self.window_size, channel)

class SimpleUNetGenerator(nn.Module):
    """Simple, robust U-Net generator without complex skip connections"""
    
    def __init__(self, input_nc=3, output_nc=3, ngf=64):
        super().__init__()
        
        # Encoder
        self.enc1 = nn.Sequential(
            nn.Conv2d(input_nc, ngf, 4, 2, 1),  # 256 -> 128
            nn.LeakyReLU(0.2, True)
        )
        
        self.enc2 = nn.Sequential(
            nn.Conv2d(ngf, ngf * 2, 4, 2, 1),  # 128 -> 64
            nn.InstanceNorm2d(ngf * 2),
            nn.LeakyReLU(0.2, True)
        )
        
        self.enc3 = nn.Sequential(
            nn.Conv2d(ngf * 2, ngf * 4, 4, 2, 1),  # 64 -> 32
            nn.InstanceNorm2d(ngf * 4),
            nn.LeakyReLU(0.2, True)
        )
        
        self.enc4 = nn.Sequential(
            nn.Conv2d(ngf * 4, ngf * 8, 4, 2, 1),  # 32 -> 16
            nn.InstanceNorm2d(ngf * 8),
            nn.LeakyReLU(0.2, True)
        )
        
        self.enc5 = nn.Sequential(
            nn.Conv2d(ngf * 8, ngf * 8, 4, 2, 1),  # 16 -> 8
            nn.InstanceNorm2d(ngf * 8),
            nn.LeakyReLU(0.2, True)
        )
        
        self.enc6 = nn.Sequential(
            nn.Conv2d(ngf * 8, ngf * 8, 4, 2, 1),  # 8 -> 4
            nn.InstanceNorm2d(ngf * 8),
            nn.LeakyReLU(0.2, True)
        )
        
        # Bottleneck
        self.bottleneck = nn.Sequential(
            nn.Conv2d(ngf * 8, ngf * 8, 4, 2, 1),  # 4 -> 2
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 8, ngf * 8, 4, 2, 1),  # 2 -> 4
            nn.InstanceNorm2d(ngf * 8),
            nn.ReLU(True)
        )
        
        # Decoder
        self.dec6 = nn.Sequential(
            nn.ConvTranspose2d(ngf * 8 * 2, ngf * 8, 4, 2, 1),  # 4 -> 8
            nn.InstanceNorm2d(ngf * 8),
            nn.Dropout2d(0.5),
            nn.ReLU(True)
        )
        
        self.dec5 = nn.Sequential(
            nn.ConvTranspose2d(ngf * 8 * 2, ngf * 8, 4, 2, 1),  # 8 -> 16
            nn.InstanceNorm2d(ngf * 8),
            nn.Dropout2d(0.5),
            nn.ReLU(True)
        )
        
        self.dec4 = nn.Sequential(
            nn.ConvTranspose2d(ngf * 8 * 2, ngf * 4, 4, 2, 1),  # 16 -> 32
            nn.InstanceNorm2d(ngf * 4),
            nn.ReLU(True)
        )
        
        self.dec3 = nn.Sequential(
            nn.ConvTranspose2d(ngf * 4 * 2, ngf * 2, 4, 2, 1),  # 32 -> 64
            nn.InstanceNorm2d(ngf * 2),
            nn.ReLU(True)
        )
        
        self.dec2 = nn.Sequential(
            nn.ConvTranspose2d(ngf * 2 * 2, ngf, 4, 2, 1),  # 64 -> 128
            nn.InstanceNorm2d(ngf),
            nn.ReLU(True)
        )
        
        self.final = nn.Sequential(
            nn.ConvTranspose2d(ngf * 2, output_nc, 4, 2, 1),  # 128 -> 256
            nn.Tanh()
        )
    
    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)    # [B, 64, 128, 128]
        e2 = self.enc2(e1)   # [B, 128, 64, 64]
        e3 = self.enc3(e2)   # [B, 256, 32, 32]
        e4 = self.enc4(e3)   # [B, 512, 16, 16]
        e5 = self.enc5(e4)   # [B, 512, 8, 8]
        e6 = self.enc6(e5)   # [B, 512, 4, 4]
        
        # Bottleneck
        b = self.bottleneck(e6)  # [B, 512, 4, 4]
        
        # Decoder with skip connections
        d6 = self.dec6(torch.cat([b, e6], 1))      # [B, 512, 8, 8]
        d5 = self.dec5(torch.cat([d6, e5], 1))     # [B, 512, 16, 16]
        d4 = self.dec4(torch.cat([d5, e4], 1))     # [B, 256, 32, 32]
        d3 = self.dec3(torch.cat([d4, e3], 1))     # [B, 128, 64, 64]
        d2 = self.dec2(torch.cat([d3, e2], 1))     # [B, 64, 128, 128]
        output = self.final(torch.cat([d2, e1], 1)) # [B, 3, 256, 256]
        
        return output

# ============================================================================
# METRICS CALCULATOR (REUSE FROM TRAINING)
# ============================================================================

class MetricsCalculator:
    """Calculate comprehensive metrics for medical images"""
    
    def __init__(self, device):
        self.device = device
    
    def calculate_psnr(self, img1, img2, max_val=2.0):
        """Calculate PSNR"""
        mse = torch.mean((img1 - img2) ** 2)
        if mse == 0:
            return torch.tensor(100.0)
        return 20 * torch.log10(max_val / torch.sqrt(mse))
    
    def calculate_rmse(self, img1, img2):
        """Calculate Root Mean Square Error"""
        mse = torch.mean((img1 - img2) ** 2)
        return torch.sqrt(mse)
    
    def calculate_nrmse(self, img1, img2):
        """Calculate Normalized Root Mean Square Error"""
        rmse = self.calculate_rmse(img1, img2)
        target_range = torch.max(img2) - torch.min(img2)
        if target_range == 0:
            return torch.tensor(0.0)
        return rmse / target_range
    
    def calculate_snr(self, img1, img2):
        """Calculate Signal-to-Noise Ratio"""
        signal_power = torch.mean(img2 ** 2)
        noise_power = torch.mean((img1 - img2) ** 2)
        if noise_power == 0:
            return torch.tensor(100.0)
        return 10 * torch.log10(signal_power / noise_power)
    
    def calculate_ssim_metric(self, img1, img2):
        """Calculate SSIM metric using skimage"""
        try:
            img1_np = img1.detach().cpu().numpy()
            img2_np = img2.detach().cpu().numpy()
            
            ssim_values = []
            for i in range(img1_np.shape[0]):
                im1 = np.transpose(img1_np[i], (1, 2, 0))
                im2 = np.transpose(img2_np[i], (1, 2, 0))
                
                # Convert to [0, 1] range
                im1 = np.clip((im1 + 1) / 2, 0, 1)
                im2 = np.clip((im2 + 1) / 2, 0, 1)
                
                ssim_val = ssim(im1, im2, multichannel=True, data_range=1.0, channel_axis=2)
                ssim_values.append(ssim_val)
            
            return np.mean(ssim_values)
        except:
            return 0.5  # Default value if calculation fails
    
    def evaluate(self, generated, target):
        """Calculate all metrics"""
        with torch.no_grad():
            metrics = {}
            
            try:
                metrics['psnr'] = self.calculate_psnr(generated, target).item()
                metrics['mae'] = F.l1_loss(generated, target).item()
                metrics['mse'] = F.mse_loss(generated, target).item()
                metrics['rmse'] = self.calculate_rmse(generated, target).item()
                metrics['nrmse'] = self.calculate_nrmse(generated, target).item()
                metrics['snr'] = self.calculate_snr(generated, target).item()
                metrics['ssim'] = self.calculate_ssim_metric(generated, target)
            except Exception as e:
                logger.warning(f"Error calculating metrics: {e}")
                metrics = {
                    'psnr': 0.0, 'mae': 1.0, 'mse': 1.0, 'rmse': 1.0, 
                    'nrmse': 1.0, 'snr': 0.0, 'ssim': 0.0
                }
            
            return metrics

# ============================================================================
# INFERENCE DATASET
# ============================================================================

class InferenceDataset(Dataset):
    """Dataset for inference"""
    
    def __init__(self, input_folder, target_folder, image_size=256):
        self.input_folder = input_folder
        self.target_folder = target_folder
        self.image_size = image_size
        
        # Find matching pairs
        self.image_pairs = self._find_image_pairs()
        
        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Lambda(lambda x: x * 2.0 - 1.0)  # Normalize to [-1, 1]
        ])
        
        self.denormalize = transforms.Lambda(lambda x: (x + 1.0) / 2.0)  # Denormalize to [0, 1]
    
    def _find_image_pairs(self):
        """Find matching input-target pairs"""
        extensions = ['*.png', '*.jpg', '*.jpeg', '*.bmp', '*.tiff']
        
        input_files = []
        for ext in extensions:
            input_files.extend(glob.glob(os.path.join(self.input_folder, ext)))
            input_files.extend(glob.glob(os.path.join(self.input_folder, ext.upper())))
        
        pairs = []
        for input_path in input_files:
            input_name = Path(input_path).stem
            
            # Find matching target
            target_path = None
            for ext in ['.png', '.jpg', '.jpeg', '.bmp', '.tiff']:
                for case_ext in [ext, ext.upper()]:
                    candidate = os.path.join(self.target_folder, input_name + case_ext)
                    if os.path.exists(candidate):
                        target_path = candidate
                        break
                if target_path:
                    break
            
            if target_path:
                pairs.append((input_path, target_path))
        
        logger.info(f"Found {len(pairs)} image pairs for inference")
        return pairs
    
    def __len__(self):
        return len(self.image_pairs)
    
    def __getitem__(self, idx):
        input_path, target_path = self.image_pairs[idx]
        
        try:
            input_img = Image.open(input_path).convert('RGB')
            target_img = Image.open(target_path).convert('RGB')
            
            input_tensor = self.transform(input_img)
            target_tensor = self.transform(target_img)
            
            return {
                'input': input_tensor,
                'target': target_tensor,
                'filename': Path(input_path).stem,
                'input_path': input_path,
                'target_path': target_path
            }
        except Exception as e:
            logger.error(f"Error loading image {idx}: {e}")
            raise

# ============================================================================
# INFERENCE CLASS (WITH FIXED MODEL LOADING)
# ============================================================================

class ModelInference:
    """Handle model inference and evaluation"""
    
    def __init__(self, model_path, device='auto'):
        self.device = self._setup_device(device)
        self.model_path = model_path
        
        # Load model
        self.generator = self._load_model()
        
        # Initialize metrics calculator
        self.metrics_calc = MetricsCalculator(self.device)
        
        logger.info("Model inference class initialized successfully")
    
    def _setup_device(self, device):
        """Setup computing device"""
        if device == 'auto':
            if torch.cuda.is_available():
                device = 'cuda'
                gpu_name = torch.cuda.get_device_name(0)
                logger.info(f"Using GPU: {gpu_name}")
            else:
                device = 'cpu'
                logger.info("Using CPU")
        else:
            logger.info(f"Using device: {device}")
        
        return torch.device(device)
    
    def _load_model(self):
        """
        Load trained model from checkpoint with flexible format support
        FIXED VERSION - Handles multiple checkpoint formats automatically
        """
        try:
            logger.info(f"Loading model from: {self.model_path}")
            
            # Load checkpoint
            checkpoint = torch.load(self.model_path, map_location=self.device)
            
            # Initialize generator
            generator = SimpleUNetGenerator().to(self.device)
            
            # Try different checkpoint formats
            state_dict = None
            checkpoint_info = {}
            
            if isinstance(checkpoint, dict):
                # Format 1: checkpoint['generator_state_dict']
                if 'generator_state_dict' in checkpoint:
                    state_dict = checkpoint['generator_state_dict']
                    logger.info("✓ Loading from format: checkpoint['generator_state_dict']")
                    
                # Format 2: checkpoint['model_state_dict']
                elif 'model_state_dict' in checkpoint:
                    state_dict = checkpoint['model_state_dict']
                    logger.info("✓ Loading from format: checkpoint['model_state_dict']")
                    
                # Format 3: checkpoint['state_dict']
                elif 'state_dict' in checkpoint:
                    state_dict = checkpoint['state_dict']
                    logger.info("✓ Loading from format: checkpoint['state_dict']")
                    
                # Format 4: checkpoint['generator']
                elif 'generator' in checkpoint:
                    state_dict = checkpoint['generator']
                    logger.info("✓ Loading from format: checkpoint['generator']")
                    
                # Format 5: checkpoint IS a dict with model weights directly
                else:
                    # Check if this looks like a state_dict
                    sample_keys = list(checkpoint.keys())[:5]
                    if any(isinstance(k, str) and ('weight' in k or 'bias' in k or 'running' in k) 
                           for k in sample_keys):
                        state_dict = checkpoint
                        logger.info("✓ Loading from format: direct state_dict (no wrapper)")
                    else:
                        logger.error(f"Unknown checkpoint format. Available keys: {list(checkpoint.keys())}")
                        raise KeyError(f"Could not find state_dict in checkpoint. Available keys: {list(checkpoint.keys())}")
                
                # Extract training info if available
                if 'epoch' in checkpoint:
                    checkpoint_info['epoch'] = checkpoint['epoch']
                if 'best_val_psnr' in checkpoint:
                    checkpoint_info['best_val_psnr'] = checkpoint['best_val_psnr']
                if 'best_val_ssim' in checkpoint:
                    checkpoint_info['best_val_ssim'] = checkpoint['best_val_ssim']
                    
            else:
                # Checkpoint is the state_dict directly (not wrapped in a dict)
                state_dict = checkpoint
                logger.info("✓ Loading from format: unwrapped state_dict")
            
            # Load the state dict into the model
            generator.load_state_dict(state_dict)
            generator.eval()
            
            # Print model info
            if checkpoint_info:
                if 'epoch' in checkpoint_info:
                    logger.info(f"Model trained for {checkpoint_info['epoch']} epochs")
                if 'best_val_psnr' in checkpoint_info:
                    logger.info(f"Best validation PSNR: {checkpoint_info['best_val_psnr']:.2f} dB")
                if 'best_val_ssim' in checkpoint_info:
                    logger.info(f"Best validation SSIM: {checkpoint_info['best_val_ssim']:.4f}")
            
            logger.info("✓ Model loaded successfully")
            return generator
            
        except Exception as e:
            logger.error(f"Failed to load model: {e}")
            logger.error(f"Full traceback:\n{traceback.format_exc()}")
            raise
    
    def save_image_comparison(self, input_img, generated_img, target_img, filename, output_dir):
        """Save comparison of input, generated, and target images"""
        try:
            # Denormalize images to [0, 1]
            input_img = np.clip((input_img + 1) / 2, 0, 1)
            generated_img = np.clip((generated_img + 1) / 2, 0, 1)
            target_img = np.clip((target_img + 1) / 2, 0, 1)
            
            # Transpose from (C, H, W) to (H, W, C)
            input_img = np.transpose(input_img, (1, 2, 0))
            generated_img = np.transpose(generated_img, (1, 2, 0))
            target_img = np.transpose(target_img, (1, 2, 0))
            
            # Create figure
            fig, axes = plt.subplots(1, 3, figsize=(15, 5))
            
            axes[0].imshow(input_img)
            axes[0].set_title('Input (Low Dose)', fontsize=12, fontweight='bold')
            axes[0].axis('off')
            
            axes[1].imshow(generated_img)
            axes[1].set_title('Generated (Model Output)', fontsize=12, fontweight='bold')
            axes[1].axis('off')
            
            axes[2].imshow(target_img)
            axes[2].set_title('Target (Full Dose)', fontsize=12, fontweight='bold')
            axes[2].axis('off')
            
            plt.tight_layout()
            
            # Save comparison image
            comparison_path = os.path.join(output_dir, 'comparisons', f'{filename}_comparison.png')
            plt.savefig(comparison_path, dpi=150, bbox_inches='tight')
            plt.close()
            
            # Save individual generated image
            generated_path = os.path.join(output_dir, 'generated', f'{filename}_generated.png')
            generated_pil = Image.fromarray((generated_img * 255).astype(np.uint8))
            generated_pil.save(generated_path)
            
        except Exception as e:
            logger.error(f"Error saving image comparison: {e}")
    
    def run_inference(self, input_folder, target_folder, output_folder):
        """Run inference on dataset"""
        try:
            # Create output directories
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            inference_dir = os.path.join(output_folder, f"inference_{timestamp}")
            
            os.makedirs(os.path.join(inference_dir, 'generated'), exist_ok=True)
            os.makedirs(os.path.join(inference_dir, 'comparisons'), exist_ok=True)
            os.makedirs(os.path.join(inference_dir, 'metrics'), exist_ok=True)
            
            logger.info(f"Output directory created: {inference_dir}")
            
            # Create dataset and dataloader
            dataset = InferenceDataset(input_folder, target_folder, IMAGE_SIZE)
            dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
            
            # Store results
            results = []
            all_metrics = {
                'psnr': [], 'ssim': [], 'mae': [], 'mse': [], 
                'rmse': [], 'nrmse': [], 'snr': []
            }
            
            logger.info(f"Processing {len(dataset)} images...")
            
            with torch.no_grad():
                for batch_idx, batch in enumerate(tqdm(dataloader, desc="Processing images")):
                    try:
                        input_img = batch['input'].to(self.device)
                        target_img = batch['target'].to(self.device)
                        filename = batch['filename'][0]
                        
                        # Generate image
                        generated_img = self.generator(input_img)
                        
                        # Calculate metrics
                        metrics = self.metrics_calc.evaluate(generated_img, target_img)
                        
                        # Save images
                        input_np = input_img[0].cpu().numpy()
                        generated_np = generated_img[0].cpu().numpy()
                        target_np = target_img[0].cpu().numpy()
                        
                        self.save_image_comparison(
                            input_np, generated_np, target_np, 
                            filename, inference_dir
                        )
                        
                        # Store results
                        result = {
                            'filename': filename,
                            'input_path': batch['input_path'][0],
                            'target_path': batch['target_path'][0],
                            **metrics
                        }
                        results.append(result)
                        
                        # Accumulate metrics
                        for key in all_metrics:
                            all_metrics[key].append(metrics[key])
                        
                        # Log progress every 10 images
                        if (batch_idx + 1) % 10 == 0:
                            logger.info(f"Processed {batch_idx + 1}/{len(dataloader)} images. "
                                      f"Avg PSNR: {np.mean(all_metrics['psnr']):.2f} dB, "
                                      f"Avg SSIM: {np.mean(all_metrics['ssim']):.3f}")
                    
                    except Exception as e:
                        logger.error(f"Error processing batch {batch_idx}: {e}")
                        continue
            
            # Calculate summary statistics
            summary_stats = {}
            for metric, values in all_metrics.items():
                if values:
                    summary_stats[metric] = {
                        'mean': np.mean(values),
                        'std': np.std(values),
                        'min': np.min(values),
                        'max': np.max(values),
                        'median': np.median(values)
                    }
            
            # Save detailed results to CSV
            self._save_results_csv(results, summary_stats, inference_dir)
            
            # Print summary
            self._print_summary(summary_stats, len(results))
            
            logger.info(f"Inference completed! Results saved to: {inference_dir}")
            return inference_dir
            
        except Exception as e:
            logger.error(f"Error during inference: {e}")
            logger.error(traceback.format_exc())
            return None
    
    def _save_results_csv(self, results, summary_stats, output_dir):
        """Save results to CSV files"""
        try:
            # Save detailed results
            detailed_csv_path = os.path.join(output_dir, 'metrics', 'detailed_results.csv')
            df_detailed = pd.DataFrame(results)
            df_detailed.to_csv(detailed_csv_path, index=False)
            logger.info(f"Detailed results saved to: {detailed_csv_path}")
            
            # Save summary statistics
            summary_csv_path = os.path.join(output_dir, 'metrics', 'summary_statistics.csv')
            
            summary_data = []
            for metric, stats in summary_stats.items():
                for stat_name, value in stats.items():
                    summary_data.append({
                        'Metric': metric.upper(),
                        'Statistic': stat_name.capitalize(),
                        'Value': value
                    })
            
            df_summary = pd.DataFrame(summary_data)
            df_summary.to_csv(summary_csv_path, index=False)
            logger.info(f"Summary statistics saved to: {summary_csv_path}")
            
            # Save metrics comparison table
            comparison_csv_path = os.path.join(output_dir, 'metrics', 'metrics_comparison.csv')
            comparison_data = []
            
            for metric, stats in summary_stats.items():
                comparison_data.append({
                    'Metric': metric.upper(),
                    'Mean': f"{stats['mean']:.4f}",
                    'Std': f"{stats['std']:.4f}",
                    'Min': f"{stats['min']:.4f}",
                    'Max': f"{stats['max']:.4f}",
                    'Median': f"{stats['median']:.4f}"
                })
            
            df_comparison = pd.DataFrame(comparison_data)
            df_comparison.to_csv(comparison_csv_path, index=False)
            logger.info(f"Metrics comparison saved to: {comparison_csv_path}")
            
        except Exception as e:
            logger.error(f"Error saving CSV files: {e}")
    
    def _print_summary(self, summary_stats, num_images):
        """Print inference summary"""
        print("\n" + "="*80)
        print("INFERENCE RESULTS SUMMARY")
        print("="*80)
        print(f"Total images processed: {num_images}")
        print("\nMETRICS SUMMARY:")
        print("-"*50)
        
        # Quality metrics (higher is better)
        print("\nQuality Metrics (Higher is Better):")
        for metric in ['psnr', 'ssim', 'snr']:
            if metric in summary_stats:
                stats = summary_stats[metric]
                unit = " dB" if metric in ['psnr', 'snr'] else ""
                print(f"  {metric.upper():8}: {stats['mean']:.4f}±{stats['std']:.4f}{unit} "
                      f"(range: {stats['min']:.4f} - {stats['max']:.4f})")
        
        # Error metrics (lower is better)
        print("\nError Metrics (Lower is Better):")
        for metric in ['mae', 'mse', 'rmse', 'nrmse']:
            if metric in summary_stats:
                stats = summary_stats[metric]
                print(f"  {metric.upper():8}: {stats['mean']:.6f}±{stats['std']:.6f} "
                      f"(range: {stats['min']:.6f} - {stats['max']:.6f})")
        
        print("="*80)

# ============================================================================
# MAIN INFERENCE FUNCTION
# ============================================================================

def run_model_inference():
    """Main function to run model inference"""
    
    print("\nEnhanced Pix2Pix Model Inference - Medical Image Translation")
    print("=" * 80)
    
    try:
        # Validate paths
        if not os.path.exists(MODEL_PATH):
            logger.error(f"Model file not found: {MODEL_PATH}")
            logger.info("Please update MODEL_PATH in the configuration section")
            return None
        
        if not os.path.exists(TEST_INPUT_FOLDER):
            logger.error(f"Input folder not found: {TEST_INPUT_FOLDER}")
            return None
        
        if not os.path.exists(TEST_TARGET_FOLDER):
            logger.error(f"Target folder not found: {TEST_TARGET_FOLDER}")
            return None
        
        logger.info("All paths validated successfully")
        logger.info(f"Model: {MODEL_PATH}")
        logger.info(f"Input: {TEST_INPUT_FOLDER}")
        logger.info(f"Target: {TEST_TARGET_FOLDER}")
        logger.info(f"Output: {OUTPUT_FOLDER}")
        
        # Initialize inference class
        inference = ModelInference(MODEL_PATH, DEVICE)
        
        # Run inference
        result_dir = inference.run_inference(
            TEST_INPUT_FOLDER, 
            TEST_TARGET_FOLDER, 
            OUTPUT_FOLDER
        )
        
        if result_dir:
            print(f"\n" + "="*80)
            print("INFERENCE COMPLETED SUCCESSFULLY!")
            print("="*80)
            print(f"Results saved to: {result_dir}")
            print("\nGenerated files:")
            print("  📁 generated/          - Individual generated images")
            print("  📁 comparisons/        - Side-by-side comparison images")
            print("  📁 metrics/            - CSV files with detailed metrics")
            print("     ├── detailed_results.csv      - Per-image metrics")
            print("     ├── summary_statistics.csv    - Statistical summary")
            print("     └── metrics_comparison.csv    - Formatted comparison table")
            print("="*80)
            return result_dir
        else:
            logger.error("Inference failed!")
            return None
            
    except Exception as e:
        logger.error(f"Inference failed with error: {e}")
        logger.error(f"Traceback: {traceback.format_exc()}")
        return None

# ============================================================================
# BATCH INFERENCE FOR MULTIPLE MODELS
# ============================================================================

def run_batch_inference(model_paths, test_input_folder, test_target_folder, output_folder):
    """Run inference for multiple models and compare results"""
    
    print("\nBatch Model Inference - Multiple Model Comparison")
    print("=" * 80)
    
    all_results = {}
    
    for model_name, model_path in model_paths.items():
        try:
            logger.info(f"\nProcessing model: {model_name}")
            logger.info(f"Model path: {model_path}")
            
            if not os.path.exists(model_path):
                logger.warning(f"Model not found: {model_path}")
                continue
            
            # Create model-specific output directory
            model_output_dir = os.path.join(output_folder, f"batch_inference_{model_name}")
            
            # Initialize inference
            inference = ModelInference(model_path, DEVICE)
            
            # Run inference
            result_dir = inference.run_inference(
                test_input_folder, 
                test_target_folder, 
                model_output_dir
            )
            
            if result_dir:
                all_results[model_name] = result_dir
                logger.info(f"✓ Completed inference for {model_name}")
            else:
                logger.error(f"✗ Failed inference for {model_name}")
                
        except Exception as e:
            logger.error(f"Error processing model {model_name}: {e}")
            continue
    
    # Create comparison summary
    if all_results:
        _create_model_comparison(all_results, output_folder)
        
        print(f"\n" + "="*80)
        print("BATCH INFERENCE COMPLETED!")
        print("="*80)
        print(f"Processed {len(all_results)} models successfully:")
        for model_name, result_dir in all_results.items():
            print(f"  ✓ {model_name}: {result_dir}")
        print(f"\nComparison summary saved to: {output_folder}")
        print("="*80)
    
    return all_results

def _create_model_comparison(all_results, output_folder):
    """Create comparison summary for multiple models"""
    try:
        comparison_data = []
        
        for model_name, result_dir in all_results.items():
            # Read summary statistics
            summary_path = os.path.join(result_dir, 'metrics', 'summary_statistics.csv')
            if os.path.exists(summary_path):
                df = pd.read_csv(summary_path)
                
                # Extract mean values for each metric
                mean_data = df[df['Statistic'] == 'Mean']
                model_row = {'Model': model_name}
                
                for _, row in mean_data.iterrows():
                    metric = row['Metric'].lower()
                    model_row[metric] = row['Value']
                
                comparison_data.append(model_row)
        
        if comparison_data:
            # Save model comparison
            comparison_df = pd.DataFrame(comparison_data)
            comparison_path = os.path.join(output_folder, 'model_comparison.csv')
            comparison_df.to_csv(comparison_path, index=False)
            
            logger.info(f"Model comparison saved to: {comparison_path}")
    
    except Exception as e:
        logger.error(f"Error creating model comparison: {e}")

# ============================================================================
# UTILITY FUNCTIONS (WITH FIXED VALIDATION)
# ============================================================================

def find_best_model(experiment_dir):
    """Find the best model checkpoint in an experiment directory"""
    try:
        checkpoints_dir = os.path.join(experiment_dir, 'checkpoints')
        
        # Look for best_model.pth first
        best_model_path = os.path.join(checkpoints_dir, 'best_model.pth')
        if os.path.exists(best_model_path):
            return best_model_path
        
        # Otherwise, find the latest checkpoint
        checkpoint_files = glob.glob(os.path.join(checkpoints_dir, 'checkpoint_epoch_*.pth'))
        
        if checkpoint_files:
            # Sort by epoch number
            def extract_epoch(path):
                filename = os.path.basename(path)
                epoch_str = filename.split('epoch_')[1].split('.pth')[0]
                return int(epoch_str)
            
            latest_checkpoint = max(checkpoint_files, key=extract_epoch)
            return latest_checkpoint
        
        return None
        
    except Exception as e:
        logger.error(f"Error finding best model: {e}")
        return None

def validate_model_checkpoint(model_path):
    """
    Validate that a model checkpoint is loadable - supports multiple formats
    FIXED VERSION
    """
    try:
        checkpoint = torch.load(model_path, map_location='cpu')
        
        # Extract state_dict
        state_dict = None
        if isinstance(checkpoint, dict):
            for key in ['generator_state_dict', 'model_state_dict', 'state_dict', 'generator']:
                if key in checkpoint:
                    state_dict = checkpoint[key]
                    logger.info(f"Found state_dict at key: '{key}'")
                    break
            if state_dict is None:
                # Check if checkpoint IS the state_dict
                if any('weight' in str(k) for k in list(checkpoint.keys())[:5]):
                    state_dict = checkpoint
                    logger.info("Checkpoint is direct state_dict")
        else:
            state_dict = checkpoint
            logger.info("Checkpoint is unwrapped state_dict")
        
        if state_dict is None:
            logger.error(f"Could not find state_dict. Keys: {list(checkpoint.keys()) if isinstance(checkpoint, dict) else 'N/A'}")
            return False
        
        # Validate by loading
        generator = SimpleUNetGenerator()
        generator.load_state_dict(state_dict)
        logger.info(f"✓ Validation successful: {model_path}")
        return True
        
    except Exception as e:
        logger.error(f"✗ Validation failed: {e}")
        logger.error(traceback.format_exc())
        return False

def create_inference_config(model_path, input_folder, target_folder, output_folder):
    """Create inference configuration and validate all paths"""
    config = {
        'model_path': model_path,
        'input_folder': input_folder,
        'target_folder': target_folder,
        'output_folder': output_folder,
        'image_size': IMAGE_SIZE,
        'batch_size': BATCH_SIZE,
        'device': DEVICE
    }
    
    # Validate all paths
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model not found: {model_path}")
    
    if not os.path.exists(input_folder):
        raise FileNotFoundError(f"Input folder not found: {input_folder}")
    
    if not os.path.exists(target_folder):
        raise FileNotFoundError(f"Target folder not found: {target_folder}")
    
    # Create output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)
    
    return config

# ============================================================================
# EXAMPLE USAGE CONFIGURATIONS
# ============================================================================

def example_single_model_inference():
    """Example configuration for single model inference"""
    
    # UPDATE THESE PATHS FOR YOUR SETUP
    model_path = "D:/Jafar/New Low-dose/CH/2(50%)/generated/experiment_20251117_134739/checkpoints/best_model.pth"
    input_folder = "D:/Jafar/New Low-dose/CH/2(50%)/test"
    target_folder = "D:/Jafar/New Low-dose/CH/Normal/test"
    output_folder = "D:/Jafar/New Low-dose/CH/2(50%)/Inference_Results"

    # Create inference instance
    inference = ModelInference(model_path, DEVICE)
    
    # Run inference
    result_dir = inference.run_inference(input_folder, target_folder, output_folder)
    
    return result_dir

def example_batch_inference():
    """Example configuration for batch inference with multiple models"""
    
    # Dictionary of model names and paths
    model_paths = {
        'best_model': "D:/Jafar/New Low-dose/CH/2(50%)/generated/experiment_20251117_134739/checkpoints/best_model.pth",
        'experiment_2': "D:/Jafar/New Low-dose/CH/2(50%)/generated/experiment_20251117_134739/checkpoints/best_model.pth"
    }
    
    input_folder = "D:/Jafar/New Low-dose/CH/2(50%)/test"
    target_folder = "D:/Jafar/New Low-dose/CH/Normal/test"
    output_folder = "D:/Jafar/New Low-dose/CH/2(50%)/Inference_Results"
    # Run batch inference
    results = run_batch_inference(model_paths, input_folder, target_folder, output_folder)
    
    return results

# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":
    print("\n" + "="*80)
    print("Enhanced Pix2Pix Model Inference Script (FIXED VERSION)")
    print("="*80)
    print("Select inference mode:")
    print("1. Single model inference")
    print("2. Batch model inference")
    print("3. Auto-find and run best model")
    
    try:
        choice = input("\nEnter your choice (1, 2, or 3): ").strip()
        
        if choice == "1":
            # Single model inference
            result = run_model_inference()
            
        elif choice == "2":
            # Batch inference example
            print("\nFor batch inference, please modify the model_paths dictionary in example_batch_inference()")
            print("Current configuration will run with example paths")
            result = example_batch_inference()
            
        elif choice == "3":
            # Auto-find best model
            experiment_dir = input("Enter experiment directory path: ").strip()
            if experiment_dir and os.path.exists(experiment_dir):
                best_model = find_best_model(experiment_dir)
                if best_model:
                    logger.info(f"Found best model: {best_model}")
                    
                    # Update global MODEL_PATH and run inference
                    MODEL_PATH = best_model
                    result = run_model_inference()
                else:
                    logger.error("No model checkpoints found in the experiment directory")
            else:
                logger.error("Invalid experiment directory")
                
        else:
            print("Invalid choice. Please run again and select 1, 2, or 3.")
            
    except KeyboardInterrupt:
        print("\nInference cancelled by user")
    except Exception as e:
        logger.error(f"Inference failed: {e}")
        print(f"\nError details: {traceback.format_exc()}")
    
    print("\nInference script completed.")
    print("="*80)


Enhanced Pix2Pix Model Inference Script (FIXED VERSION)
Select inference mode:
1. Single model inference
2. Batch model inference
3. Auto-find and run best model



Enter your choice (1, 2, or 3):  1


2025-11-23 08:19:16,561 - INFO - All paths validated successfully
2025-11-23 08:19:16,561 - INFO - Model: D:/Jafar/New Low-dose/CH/2(50%)/generated/experiment_20251117_134739/checkpoints/best_model.pth
2025-11-23 08:19:16,561 - INFO - Input: D:/Jafar/New Low-dose/CH/2(50%)/test
2025-11-23 08:19:16,561 - INFO - Target: D:/Jafar/New Low-dose/CH/Normal/test
2025-11-23 08:19:16,561 - INFO - Output: D:/Jafar/New Low-dose/CH/2(50%)/Inference_Results
2025-11-23 08:19:16,611 - INFO - Using GPU: NVIDIA GeForce RTX 4080
2025-11-23 08:19:16,612 - INFO - Loading model from: D:/Jafar/New Low-dose/CH/2(50%)/generated/experiment_20251117_134739/checkpoints/best_model.pth



Enhanced Pix2Pix Model Inference - Medical Image Translation


2025-11-23 08:19:17,740 - INFO - ✓ Loading from format: checkpoint['generator']
2025-11-23 08:19:17,752 - INFO - Model trained for 131 epochs
2025-11-23 08:19:17,752 - INFO - Best validation PSNR: 48.35 dB
2025-11-23 08:19:17,753 - INFO - ✓ Model loaded successfully
2025-11-23 08:19:17,755 - INFO - Model inference class initialized successfully
2025-11-23 08:19:17,757 - INFO - Output directory created: D:/Jafar/New Low-dose/CH/2(50%)/Inference_Results\inference_20251123_081917
2025-11-23 08:19:17,817 - INFO - Found 1000 image pairs for inference
2025-11-23 08:19:17,817 - INFO - Processing 1000 images...
Processing images: 100%|██████████| 1000/1000 [09:32<00:00,  1.75it/s]
2025-11-23 08:28:50,043 - INFO - Detailed results saved to: D:/Jafar/New Low-dose/CH/2(50%)/Inference_Results\inference_20251123_081917\metrics\detailed_results.csv
2025-11-23 08:28:50,043 - INFO - Summary statistics saved to: D:/Jafar/New Low-dose/CH/2(50%)/Inference_Results\inference_20251123_081917\metrics\summary


INFERENCE RESULTS SUMMARY
Total images processed: 1000

METRICS SUMMARY:
--------------------------------------------------

Quality Metrics (Higher is Better):
  PSNR    : 50.0649±2.5206 dB (range: 41.9358 - 53.9462)
  SSIM    : 0.9975±0.0009 (range: 0.9951 - 0.9988)
  SNR     : 43.8515±2.5423 dB (range: 35.7030 - 47.7878)

Error Metrics (Lower is Better):
  MAE     : 0.002193±0.000693 (range: 0.001395 - 0.007415)
  MSE     : 0.000049±0.000042 (range: 0.000016 - 0.000256)
  RMSE    : 0.006584±0.002338 (range: 0.004015 - 0.016004)
  NRMSE   : 0.006032±0.002448 (range: 0.002539 - 0.016481)

INFERENCE COMPLETED SUCCESSFULLY!
Results saved to: D:/Jafar/New Low-dose/CH/2(50%)/Inference_Results\inference_20251123_081917

Generated files:
  📁 generated/          - Individual generated images
  📁 comparisons/        - Side-by-side comparison images
  📁 metrics/            - CSV files with detailed metrics
     ├── detailed_results.csv      - Per-image metrics
     ├── summary_statistics.csv 